In [ ]:
# --- SETUP ---
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", device)

Running on: cpu


In [16]:
class MLP(nn.Module):
    # randomly drops 30% of neurons during training to keep the model from overfitting
    def __init__(self, in_dim, hidden=[512,256,128], dropout=0.3):
        super().__init__()
        layers = []
        last = in_dim
        for h in hidden:
            layers += [nn.Linear(last, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            last = h
        layers.append(nn.Linear(last, 2))  # Binary classification
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

In [17]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds, probs = [], []
    for (xb,) in loader:          
        xb = xb.to(device)
        out = model(xb)       
        preds.append(torch.argmax(out, dim=1).cpu())
    preds = torch.cat(preds).numpy()
    return preds



In [18]:

for n in [10, 12, 14, 16, 18, 20]:
    selected_n_value = n
    kryptonite_model_results = torch.load(f'models/results_{selected_n_value}', weights_only=False)
    X_new = np.load('datasets/hidden-kryptonite-%s-X.npy'%(selected_n_value))
    X_new = torch.tensor(X_new, dtype=torch.float32)
    new_dl = DataLoader(TensorDataset(X_new), batch_size=256)

    best_idx = kryptonite_model_results['best_idx']
    best_result = kryptonite_model_results['results'][best_idx]
    
    cfg = best_result['config']
    state = kryptonite_model_results['best_state_dict']

    # rebuild the model architecture
    best_model = MLP(
        in_dim=cfg['in_dim'],
        hidden=cfg['hidden'],
        dropout=cfg['dropout']
    ).to(device)

    best_model.load_state_dict(state);

    # run predictions
    y_hat = predict(best_model, new_dl)
    assert y_hat.shape == (10000,) and set(np.unique(y_hat)) <= {0,1}, \
       f"y_hat failed checks: shape={y_hat.shape}, unique={np.unique(y_hat)}"

    np.save(f'hiddenlabels/y_predicted_{selected_n_value}.npy', y_hat)
    print(f'hiddenlabels/y_predicted_{selected_n_value}.npy saved correctly and checked formatting')

hiddenlabels/y_predicted_10.npy saved correctly and checked formatting
hiddenlabels/y_predicted_12.npy saved correctly and checked formatting
hiddenlabels/y_predicted_14.npy saved correctly and checked formatting
hiddenlabels/y_predicted_16.npy saved correctly and checked formatting
hiddenlabels/y_predicted_18.npy saved correctly and checked formatting
hiddenlabels/y_predicted_20.npy saved correctly and checked formatting
